<a href="https://colab.research.google.com/github/patriciarrs/Rag-finalproject/blob/main/v5/RAG_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file (the student portal doesn't accept .ipynb files)

## Mandatory
- Ingestion Phase
- Inference Phase

# 2. Project Specific Information

## Project Description
This RAG system allows users to ask questions about the Bitcoin and Ethereum whitepapers.
It loads the PDFs, chunks and embeds the text into a vector store, and answers questions
by retrieving the most relevant passages and generating a response with an LLM.
v2 adds a Gradio chat interface so users can interact with the system without writing any code.
v3 adds RAGAS evaluation to measure retrieval and answer quality with objective metrics.
v4 adds multi-query retrieval and FlashRank reranking, with a RAGAS before/after comparison.
v5 adds adaptive retrieval: the system runs the cheap baseline first and only escalates to
multi-query + reranking when the baseline returns low-confidence results, reducing cost
without sacrificing quality on hard questions.

## Steps
1. Load Bitcoin and Ethereum whitepapers (PDFs)
2. Clean and preprocess the text
3. Split documents into chunks
4. Embed chunks and store in Pinecone
5. Build three retrieval strategies: baseline, multi-query + reranking, and adaptive
6. Support multi-turn conversations via chat history
7. Launch a Gradio chat interface with strategy selector
8. Test all three strategies with example queries
9. Compare all three strategies using RAGAS

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [x] Text preprocessing (clean PDF noise)
- [x] Auto topic detection via LLM
- [x] Chat history support (last 5 turns)
- [x] Gradio chat interface with strategy selector
- [x] RAGAS evaluation
- [x] Multi-query retrieval
- [x] FlashRank reranking
- [x] Before/after strategy comparison with RAGAS
- [x] Adaptive retrieval (confidence-based escalation)

# 3. Installation

In [ ]:
# Install all required packages

# ⚠️ IMPORTANT — READ BEFORE RUNNING
# After this cell completes, you must manually restart the runtime:
#   Runtime → Restart session
# Then run all cells from Section 4 onwards (skip re-running this cell).
# Skipping the restart may cause this error on import:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility
# If that happens: Runtime → Restart session → re-run from Section 4.

# Pin numpy first to avoid binary incompatibility errors.
# Some packages (ragas, datasets) are compiled against numpy <2.0.
# Installing this first ensures all packages share the same numpy build.
%pip install "numpy<2.0" -qqq

%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-pinecone==0.2.0" -qqq
%pip install "pinecone==5.4.2" -qqq
%pip install pypdf -qqq
%pip install gradio -qqq
%pip install ragas -qqq
%pip install datasets -qqq
%pip install flashrank -qqq

print("✓ Installation complete. Go to Runtime → Restart session, then run from Section 4.")

# 4. Setup

In [ ]:
# ─────────────────────────────────────────────────────────────
# load_secrets() works in two environments:
#   - Google Colab  → reads from Colab Secrets (sidebar 🔑)
#   - Local machine → reads from a .env file in the project root
# ─────────────────────────────────────────────────────────────
def load_secrets(keys):
    import os

    try:
        # Try Colab first
        from google.colab import userdata
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Fall back to .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    # Check that all required keys are present
    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            raise ValueError(
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )

    # Expose all keys as environment variables so libraries can pick them up automatically
    for key, value in values.items():
        os.environ[key] = value

    return values

In [ ]:
# Load API keys
# Make sure PINECONE_API_KEY and OPENAI_API_KEY
# are set in Colab Secrets (or your local .env file)
try:
    secrets = load_secrets(['PINECONE_API_KEY', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)

# 5. Global Variables

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Models ────────────────────────────────────────────────────
# Embeddings: converts text chunks into vector representations
# text-embedding-3-small produces 1536-dimensional vectors
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Main LLM: generates answers based on retrieved context
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Lightweight LLM for classification tasks (cheaper + faster)
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0   # temperature=0 → deterministic output (good for classification)
)

# ── Pinecone Setup ────────────────────────────────────────────
# Pinecone is a managed cloud vector database.
# We create the index once; if it already exists, this is skipped.
INDEX_NAME = "crypto-whitepapers"
EMBEDDING_DIMENSIONS = 1536   # Must match text-embedding-3-small output size

# Initialize Pinecone client with our API key
pc = Pinecone(api_key=secrets["PINECONE_API_KEY"])

# Create the index if it doesn't exist yet
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",              # Cosine similarity is standard for text embeddings
        spec=ServerlessSpec(
            cloud="aws",              # Pinecone free tier runs on AWS
            region="us-east-1"
        )
    )
    print(f"✓ Created Pinecone index: '{INDEX_NAME}'")
else:
    print(f"✓ Pinecone index '{INDEX_NAME}' already exists")

# Connect LangChain to our Pinecone index
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=secrets["PINECONE_API_KEY"]
)

# ── Text Splitter ─────────────────────────────────────────────
# Splits large documents into smaller overlapping chunks.
# chunk_size=1000: ~1000 characters per chunk (fits well in LLM context)
# chunk_overlap=200: 20% overlap so sentences aren't cut off between chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# 6. Indexing / Ingestion Phase

In [ ]:
import re

def clean_text(text: str) -> str:
    """
    TEXT PREPROCESSING - Remove noise from raw PDF text.

    PDFs often contain extra whitespace, page numbers, and
    other artifacts that hurt retrieval quality if left in.

    Args:
        text (str): Raw text extracted from a PDF page

    Returns:
        str: Cleaned text
    """
    # Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove standalone page numbers that appear at the end of lines
    text = re.sub(r'(?<=\.)\s*\d+\s*$', '', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(documents: list) -> str:
    """
    AUTO-DETECT TOPIC - Use an LLM to figure out what the document is about.

    We sample the first 3 pages and ask the LLM for a one-word topic.
    This gets stored as metadata on each chunk so we can filter later.

    Args:
        documents (list): List of Document objects loaded from a PDF

    Returns:
        str: Detected topic (e.g., 'bitcoin', 'ethereum')
    """

    # Prompt that asks for a single-word topic label
    topic_detection_template = ChatPromptTemplate.from_template(
        """
        Analyze the following document content and determine its primary topic.

        Document content:
        {content}

        Based on this content, what is the primary topic?
        Answer with a single word or short phrase (e.g., 'bitcoin', 'ethereum').

        Primary topic:
        """
    )

    # Chain: prompt → classification LLM → plain string output
    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Sample only the first 3 pages to keep costs low
    sample_content = ""
    for doc in documents[:3]:
        sample_content += doc.page_content + "\n\n"
    sample_content = sample_content[:4000]  # Cap at 4000 chars

    detected_topic = topic_detection_chain.invoke({"content": sample_content}).strip().lower()

    return detected_topic

In [ ]:
from langchain.document_loaders import PyPDFLoader

def ingest_documents(document_path: str):
    """
    INGESTION PIPELINE - Load, clean, chunk, and store a PDF.

    Steps:
      1. Load PDF pages
      2. Auto-detect topic
      3. Clean text
      4. Add metadata
      5. Split into chunks
      6. Store in Pinecone

    Args:
        document_path (str): Local file path or URL to a PDF
    """

    print("-" * 60)
    print(f"INGESTING: {document_path}")
    print("-" * 60)

    # Step 1: Load - PyPDFLoader extracts text page by page
    print("\n[1/5] Loading PDF...")
    loader = PyPDFLoader(document_path)
    documents = loader.load()
    print(f"  ✓ Loaded {len(documents)} pages")

    # Step 2: Auto-detect topic using LLM
    print("\n[2/5] Detecting topic...")
    topic = detect_document_topic(documents)
    print(f"  ✓ Topic: '{topic}'")

    # Step 3: Clean text on every page
    print("\n[3/5] Cleaning text...")
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
    print(f"  ✓ Cleaned {len(documents)} pages")

    # Step 4: Add topic as metadata so we can filter by it later
    print("\n[4/5] Adding metadata...")
    for doc in documents:
        doc.metadata["topic"] = topic
    print(f"  ✓ Metadata added")

    # Step 5: Split pages into smaller overlapping chunks
    print("\n[5/5] Chunking and storing...")
    chunks = splitter.split_documents(documents)

    # Step 6: Store chunks in Pinecone (embeddings are computed automatically)
    vectorstore.add_documents(chunks)
    print(f"  ✓ Stored {len(chunks)} chunks in Pinecone")

    print(f"\n✓ Done! '{topic}' ingested successfully.\n")

In [ ]:
# Ingest both whitepapers
# The Bitcoin whitepaper is publicly available as a URL.
# Download the Ethereum PDF and place it in the same folder as this notebook.
ingest_documents("https://bitcoin.org/bitcoin.pdf")
ingest_documents("./ethereum.pdf")

# 7. Inference Phase (RAG)

In [ ]:
def format_chat_history(history: list, max_turns: int = 5) -> str:
    """
    FORMAT CHAT HISTORY - Convert conversation history into a readable string.

    We limit to the last N turns to:
      - Stay within the LLM's context window
      - Keep costs low (fewer tokens = cheaper)
      - Focus on recent, relevant context

    Args:
        history (list): List of {role, content} message dicts
        max_turns (int): How many past turns to include

    Returns:
        str: Formatted conversation string
    """
    if not history:
        return "No previous conversation."

    # Each turn = 1 user message + 1 assistant message = 2 entries
    recent = history[-(max_turns * 2):]

    formatted = []
    for message in recent:
        role = message["role"].capitalize()
        content = message["content"]
        formatted.append(f"{role}: {content}")

    return "\n".join(formatted)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain():
    """
    CREATE RAG CHAIN - Build the prompt + LLM pipeline.

    The chain takes three inputs:
      - context: retrieved document chunks
      - history: recent conversation turns
      - query: the user's current question

    Returns:
        RunnableSequence: An LCEL chain (prompt | llm | parser)
    """

    rag_template = ChatPromptTemplate.from_template(
        """
        You are a helpful assistant answering questions about cryptocurrency whitepapers.

        Use the conversation history and provided documents to answer the current question.
        If the question references previous context (e.g., "it", "this", "that"),
        use the conversation history to understand what is being referenced.

        Instructions:
        - Answer clearly and concisely (max 2-3 paragraphs)
        - Use only the provided documents as context
        - If the answer is not in the documents, say "I don't have enough information to answer that question"

        Conversation History:
        {history}

        Documents:
        {context}

        Current Question: {query}

        Answer:
        """
    )

    # LCEL chain: prompt → LLM → plain string
    rag_chain = rag_template | llm | StrOutputParser()

    return rag_chain

In [ ]:
def ask(query: str, chat_history: list = None) -> str:
    """
    ASK - Run a single question through the RAG pipeline.

    Steps:
      1. Format chat history
      2. Retrieve the 3 most relevant chunks from Pinecone
      3. Pass chunks + history + query to the LLM
      4. Return the generated answer

    Args:
        query (str): The user's question
        chat_history (list): Previous conversation turns (optional)

    Returns:
        str: The LLM-generated answer
    """

    # Default to empty history if none provided
    if chat_history is None:
        chat_history = []

    # Step 1: Format history into a readable string
    formatted_history = format_chat_history(chat_history, max_turns=5)

    # Step 2: Retrieve top-3 relevant chunks using similarity search
    # k=3 is a good default: enough context without overwhelming the LLM
    results = vectorstore.similarity_search(query, k=3)

    # Step 3: Join chunks into a single context string
    context = "\n\n".join([doc.page_content for doc in results])

    # Step 4: Generate answer
    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    return response

In [ ]:
#──────────────────────────────────────────────────────────────
# NEW in v4: Multi-query + reranking inference
#──────────────────────────────────────────────────────────────

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

def ask_multiquery(query: str, chat_history: list = None) -> str:
    """
    MULTI-QUERY + RERANKING INFERENCE

    Improves on the baseline in two ways:

    1. Multi-query: the LLM generates 3 variations of the original question
       and searches Pinecone with each. This catches relevant chunks that
       the original phrasing might have missed.

    2. Reranking: FlashRank uses a cross-encoder model to re-score all
       retrieved chunks by relevance and keeps only the top 4.
       Cross-encoders are more accurate than cosine similarity alone
       because they compare the query and chunk together, not separately.

    Args:
        query (str): The user's question
        chat_history (list): Previous conversation turns (optional)

    Returns:
        str: The LLM-generated answer
    """

    if chat_history is None:
        chat_history = []

    formatted_history = format_chat_history(chat_history, max_turns=5)

    # Step 1: Base retriever — searches Pinecone with k=5 per query variation
    base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Step 2: Multi-query — LLM generates query variations, results are deduplicated
    multiquery_retriever = MultiQueryRetriever.from_llm(
        retriever=base_retriever,
        llm=llm
    )

    # Step 3: Reranking — FlashRank cross-encoder scores each chunk against the query
    # top_n=4: keep the 4 most relevant chunks after reranking
    compressor = FlashrankRerank(top_n=4)

    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=multiquery_retriever
    )

    results = compression_retriever.invoke(query)

    # Format context and generate answer
    context = "\n\n".join([doc.page_content for doc in results])

    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    return response

In [ ]:
#──────────────────────────────────────────────────────────────
# NEW in v5: Adaptive retrieval
#──────────────────────────────────────────────────────────────

# Confidence threshold for escalation.
# Pinecone cosine similarity scores range from 0 to 1 — higher means more similar.
# If the best-matching chunk scores below this value, the baseline is not confident
# enough and we escalate to multi-query + reranking.
# 0.75 is a reasonable default: high enough to catch genuinely weak matches,
# low enough to avoid escalating on questions the baseline handles well.
CONFIDENCE_THRESHOLD = 0.75

def ask_adaptive(
    query: str,
    chat_history: list = None,
    threshold: float = CONFIDENCE_THRESHOLD
) -> str:
    """
    ADAPTIVE RETRIEVAL

    Balances cost and quality by choosing the retrieval strategy dynamically:

    1. Run baseline similarity search and inspect the top confidence score.
    2. If score >= threshold → baseline results are good enough, use them.
    3. If score < threshold  → escalate to multi-query + reranking.

    This avoids paying the cost of multi-query (extra LLM calls to generate
    query variations) on every question — only questions where the baseline
    struggles actually trigger the more expensive pipeline.

    Args:
        query (str): The user's question
        chat_history (list): Previous conversation turns (optional)
        threshold (float): Confidence cutoff for escalation (default 0.75)

    Returns:
        str: The LLM-generated answer
    """

    if chat_history is None:
        chat_history = []

    formatted_history = format_chat_history(chat_history, max_turns=5)

    # Step 1: Baseline search with scores
    # similarity_search_with_score returns (Document, score) pairs
    results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

    # Step 2: Check top confidence score
    top_score = results_with_scores[0][1] if results_with_scores else 0.0

    if top_score >= threshold:
        # Baseline is confident — use it as-is (cheap path)
        print(f"  [Adaptive] Score: {top_score:.3f} ≥ {threshold} → Baseline")
        results = [doc for doc, score in results_with_scores]
        context = "\n\n".join([doc.page_content for doc in results])

    else:
        # Low confidence — escalate to multi-query + reranking (expensive path)
        print(f"  [Adaptive] Score: {top_score:.3f} < {threshold} → Escalating to Multi-query + Reranking")

        base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        multiquery_retriever = MultiQueryRetriever.from_llm(
            retriever=base_retriever,
            llm=llm
        )
        compressor = FlashrankRerank(top_n=4)
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=multiquery_retriever
        )
        results = compression_retriever.invoke(query)
        context = "\n\n".join([doc.page_content for doc in results])

    # Step 3: Generate answer with whichever context was retrieved
    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    return response

# 8. User Interface (Gradio)

Gradio creates an interactive chat interface directly inside Colab.
The UI handles conversation history automatically — no manual history management needed.

In [ ]:
import gradio as gr

def gradio_chat(message: str, history: list, strategy: str) -> str:
    """
    GRADIO CHAT HANDLER

    Routes the query to the selected retrieval strategy.
    The strategy dropdown value is passed in as a parameter.

    Args:
        message (str): The user's latest message
        history (list): Gradio conversation history (list of [user, assistant] pairs)
        strategy (str): Selected retrieval strategy

    Returns:
        str: The assistant's reply
    """

    # Convert Gradio's [user, assistant] pairs into our {role, content} format
    chat_history = []
    for user_msg, assistant_msg in history:
        chat_history.append({"role": "user", "content": user_msg})
        chat_history.append({"role": "assistant", "content": assistant_msg})

    # Route to the correct inference function based on selected strategy
    if strategy == "Multi-query + Reranking":
        return ask_multiquery(message, chat_history=chat_history)
    elif strategy == "Adaptive":
        return ask_adaptive(message, chat_history=chat_history)
    else:
        return ask(message, chat_history=chat_history)


# ── Build the Gradio interface ─────────────────────────────────
# Additional inputs let the user switch strategy without restarting
strategy_selector = gr.Dropdown(
    choices=["Baseline", "Multi-query + Reranking", "Adaptive"],
    value="Adaptive",   # Default to Adaptive in v5
    label="Retrieval strategy"
)

demo = gr.ChatInterface(
    fn=gradio_chat,
    additional_inputs=[strategy_selector],
    title="Crypto Whitepapers Q&A",
    description="Ask questions about the Bitcoin and Ethereum whitepapers. Switch retrieval strategy to compare results.",
    examples=[
        ["What is Bitcoin?", "Baseline"],
        ["How does proof-of-work prevent double spending?", "Adaptive"],
        ["What is Ethereum and how is it different from Bitcoin?", "Adaptive"],
        ["What is the incentive for miners to support the network?", "Multi-query + Reranking"]
    ],
    cache_examples=False
)

# share=True creates a public link so you can share the demo outside Colab
demo.launch(share=True)

# 9. Testing / Demo

In [ ]:
# ── Test all three strategies side by side ────────────────────

test_questions = [
    "What is Bitcoin?",
    "How does proof-of-work prevent double spending?",
    "What is Ethereum and how is it different from Bitcoin?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"\n[Baseline]")
    print(ask(q))
    print(f"\n[Multi-query + Reranking]")
    print(ask_multiquery(q))
    print(f"\n[Adaptive]")
    print(ask_adaptive(q))
    print("-" * 60)

In [ ]:
# ── Multi-turn conversation demo ──────────────────────────────
# Shows that the system can maintain context across follow-up questions.

# We build the history list manually to simulate a conversation
history = []

def chat(query: str):
    """Send a message and update the conversation history."""
    answer = ask(query, chat_history=history)

    # Append both sides of the turn to history
    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": answer})

    print(f"User: {query}")
    print(f"Assistant: {answer}")
    print("-" * 60)

# Simulate a conversation with follow-up questions
chat("What is Bitcoin?")
chat("How does it prevent double spending?")   # 'it' refers to Bitcoin — tests history
chat("Who gets the mining reward?")            # follow-up on the same topic

# 10. RAGAS Evaluation

RAGAS (Retrieval Augmented Generation Assessment) measures RAG quality with four metrics:

- **Faithfulness** — is the answer grounded in the retrieved context, or is the LLM hallucinating?
- **Answer Relevancy** — does the answer actually address what was asked?
- **Context Precision** — are the most relevant chunks ranked at the top of the results?
- **Context Recall** — did we retrieve all the information needed to answer correctly?

All scores range from 0 to 1. Higher is better.

> **Cost note:** RAGAS makes LLM calls for each question × metric. With 6 questions this takes ~2-3 minutes and costs roughly $0.01-0.02.

In [ ]:
# ── Evaluation Dataset ─────────────────────────────────────────
# Each item has a question and a ground_truth (the ideal answer).
# RAGAS uses the ground_truth to calculate Context Recall —
# it checks whether the retrieved chunks contain enough information
# to produce the correct answer.

EVALUATION_DATASET = [
    {
        "question": "What is Bitcoin?",
        "ground_truth": "Bitcoin is a decentralized digital currency that lets people transfer value "
                        "directly to one another over the internet, without relying on banks or other "
                        "central intermediaries. It uses a peer-to-peer network and a shared ledger "
                        "secured by cryptography and proof-of-work to record and order transactions."
    },
    {
        "question": "How does proof-of-work prevent double spending?",
        "ground_truth": "Proof-of-work makes altering transaction history extremely costly. Nodes accept "
                        "as valid the chain with the greatest accumulated work. For an attacker to spend "
                        "the same coins twice, they would need to rebuild the block containing the original "
                        "payment and all subsequent blocks, and then produce a longer chain than the honest "
                        "network, which becomes increasingly impractical as more blocks are added."
    },
    {
        "question": "What is the purpose of the timestamp server in Bitcoin?",
        "ground_truth": "The timestamp server gives transactions an ordered, verifiable place in history. "
                        "It groups data into blocks, hashes each block, and publishes the hash. Each new "
                        "block's hash includes the previous block's hash, creating a chain. This structure "
                        "proves that the recorded data existed at or before the time it was timestamped and "
                        "prevents past entries from being changed without redoing the work for all following blocks."
    },
    {
        "question": "How are transactions verified in the Bitcoin network?",
        "ground_truth": "Transactions are validated collectively by the network. Miners collect pending "
                        "transactions into a candidate block, check that each transaction follows the protocol "
                        "rules and does not reuse already spent outputs, and then perform proof-of-work on that block. "
                        "Once a miner finds a valid proof, it broadcasts the block; other full nodes accept it only if "
                        "the block itself and all included transactions are valid and unspent."
    },
    {
        "question": "What is the incentive for nodes to support the Bitcoin network?",
        "ground_truth": "Mining nodes are incentivized through economic rewards embedded in the protocol. "
                        "Each mined block contains a special transaction that creates new bitcoins for the block creator "
                        "(the block reward), and miners can also claim any difference between the total transaction inputs "
                        "and outputs as transaction fees. Together, these rewards motivate miners to spend resources securing the network."
    },
    {
        "question": "What is Ethereum and how is it different from Bitcoin?",
        "ground_truth": "Ethereum is a next-generation blockchain platform that goes beyond Bitcoin's "
                        "use as a digital currency. While Bitcoin is primarily a peer-to-peer payment "
                        "system, Ethereum introduces a Turing-complete programming environment that "
                        "enables smart contracts and decentralized applications. It includes its own "
                        "built-in currency called ether, which is used to pay for transaction fees "
                        "and computational work on the network."
    },
]

print(f"Evaluation dataset ready: {len(EVALUATION_DATASET)} questions")

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

def run_ragas_evaluation(dataset: list, retrieval_fn, strategy_name: str) -> pd.DataFrame:
    """
    RUN RAGAS EVALUATION

    Refactored in v4 to accept any retrieval function, so we can run
    the same evaluation on both strategies and compare results.

    Args:
        dataset (list): List of {question, ground_truth} dicts
        retrieval_fn (callable): The inference function to evaluate
        strategy_name (str): Label for printing (e.g. 'Baseline')

    Returns:
        pd.DataFrame: Per-question scores
    """

    print("=" * 60)
    print(f"EVALUATING: {strategy_name}")
    print("=" * 60)

    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for i, item in enumerate(dataset):
        question = item["question"]
        print(f"\n[{i+1}/{len(dataset)}] {question}")

        # Retrieve chunks using the provided function
        # We call the underlying retriever directly to get contexts for RAGAS
        if retrieval_fn == ask:
            # Baseline: simple similarity search
            results = vectorstore.similarity_search(question, k=3)
            retrieved_contexts = [doc.page_content for doc in results]
        elif retrieval_fn == ask_adaptive:
            # Adaptive: check confidence, escalate if below threshold
            results_with_scores = vectorstore.similarity_search_with_score(question, k=3)
            top_score = results_with_scores[0][1] if results_with_scores else 0.0
            if top_score >= CONFIDENCE_THRESHOLD:
                results = [doc for doc, score in results_with_scores]
                print(f"    → Baseline (score: {top_score:.3f})")
            else:
                base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
                mq_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)
                compressor = FlashrankRerank(top_n=4)
                compression_retriever = ContextualCompressionRetriever(
                    base_compressor=compressor,
                    base_retriever=mq_retriever
                )
                results = compression_retriever.invoke(question)
                print(f"    → Escalated (score: {top_score:.3f})")
            retrieved_contexts = [doc.page_content for doc in results]
        else:
            # Multi-query + reranking
            base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
            mq_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)
            compressor = FlashrankRerank(top_n=4)
            compression_retriever = ContextualCompressionRetriever(
                base_compressor=compressor,
                base_retriever=mq_retriever
            )
            results = compression_retriever.invoke(question)
            retrieved_contexts = [doc.page_content for doc in results]

        context_str = "\n\n".join(retrieved_contexts)

        # Generate answer
        answer = create_rag_chain().invoke({
            "context": context_str,
            "query": question,
            "history": "No previous conversation."
        })

        questions.append(question)
        answers.append(answer)
        contexts.append(retrieved_contexts)
        ground_truths.append(item["ground_truth"])
        print(f"  ✓ Done")

    print("\n" + "=" * 60)
    print("Running RAGAS metrics...")
    print("=" * 60 + "\n")

    ragas_dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    })

    scores = evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        embeddings=embeddings,
        llm=llm
    )

    df = scores.to_pandas()

    print(f"  Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy:  {df['answer_relevancy'].mean():.3f}")
    print(f"  Context Precision: {df['context_precision'].mean():.3f}")
    print(f"  Context Recall:    {df['context_recall'].mean():.3f}")

    return df

In [ ]:
# ── Three-way strategy comparison ─────────────────────────────
# Runs RAGAS on all three strategies and prints a side-by-side summary.
# This takes ~8-10 minutes (three full evaluation runs).

# Baseline
baseline_df = run_ragas_evaluation(EVALUATION_DATASET, ask, "Baseline")
baseline_df.insert(0, "question", [item["question"] for item in EVALUATION_DATASET])

# Multi-query + Reranking
multiquery_df = run_ragas_evaluation(EVALUATION_DATASET, ask_multiquery, "Multi-query + Reranking")
multiquery_df.insert(0, "question", [item["question"] for item in EVALUATION_DATASET])

# Adaptive
adaptive_df = run_ragas_evaluation(EVALUATION_DATASET, ask_adaptive, "Adaptive")
adaptive_df.insert(0, "question", [item["question"] for item in EVALUATION_DATASET])

# ── Summary comparison table ───────────────────────────────────
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

comparison = pd.DataFrame({
    "Metric": metrics,
    "Baseline": [baseline_df[m].mean() for m in metrics],
    "Multi-query + Reranking": [multiquery_df[m].mean() for m in metrics],
    "Adaptive": [adaptive_df[m].mean() for m in metrics],
})

# Delta columns show improvement of each strategy over baseline
comparison["Δ Multi-query"] = comparison["Multi-query + Reranking"] - comparison["Baseline"]
comparison["Δ Adaptive"] = comparison["Adaptive"] - comparison["Baseline"]

# Format for display
for col in ["Baseline", "Multi-query + Reranking", "Adaptive"]:
    comparison[col] = comparison[col].apply(lambda x: f"{x:.3f}")
for col in ["Δ Multi-query", "Δ Adaptive"]:
    comparison[col] = comparison[col].apply(lambda x: f"+{x:.3f}" if x >= 0 else f"{x:.3f}")

print("\n" + "=" * 70)
print("STRATEGY COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False))
print("=" * 70)